In [1]:
import pandas as pd
import numpy as np

In [13]:
df = pd.read_csv("../data/processed/cleaned_music_dataset.csv")
df.head()

,Unnamed: 0,artist_name,track_name,release_date,genre,lyrics,len,dating,violence,world/life,...,sadness,feelings,danceability,loudness,acousticness,instrumentalness,valence,energy,topic,age
0,0,mukesh,mohabbat bhi jhoothi,1950,pop,hold time feel break feel untrue convince spea...,95,0.000598,0.063746,0.000598,...,0.380299,0.117175,0.357739,0.454119,0.997992,0.901822,0.339448,0.137110,sadness,1.0
1,4,frankie laine,i believe,1950,pop,believe drop rain fall grow believe darkest ni...,51,0.035537,0.096777,0.443435,...,0.001284,0.001284,0.331745,0.647540,0.954819,0.000002,0.325021,0.263240,world/life,1.0
2,6,johnnie ray,cry,1950,pop,sweetheart send letter goodbye secret feel bet...,24,0.002770,0.002770,0.002770,...,0.002770,0.225422,0.456298,0.585288,0.840361,0.000000,0.351814,0.139112,music,1.0
3,10,pérez prado,patricia,1950,pop,kiss lips want stroll charm mambo chacha merin...,54,0.048249,0.001548,0.001548,...,0.225889,0.001548,0.686992,0.744404,0.083935,0.199393,0.775350,0.743736,romantic,1.0
4,12,giorgos papadopoulos,apopse eida oneiro,1950,pop,till darling till matter know till dream live ...,48,0.001350,0.001350,0.417772,...,0.068800,0.001350,0.291671,0.646489,0.975904,0.000246,0.597073,0.394375,romantic,1.0


In [14]:
num_users = 1000

user_ids = np.random.randint(1, num_users, size=len(df))

ratings = np.random.randint(1, 6, size=len(df))

user_data = pd.DataFrame({
    'user_id': user_ids,
    'track_name': df['track_name'],
    'rating': ratings
})

user_data.head()

,user_id,track_name,rating
0,778,mohabbat bhi jhoothi,5
1,274,i believe,2
2,667,cry,2
3,271,patricia,2
4,268,apopse eida oneiro,3


In [15]:
user_data.to_csv("../data/user_ratings.csv", index=False)

In [16]:
import numpy as np

print(np.__version__)

1.26.4


In [17]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse

In [18]:
ratings = pd.read_csv("../data/user_ratings.csv")

reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    ratings[['user_id', 'track_name', 'rating']],
    reader
)

In [19]:
trainset, testset = train_test_split(data, test_size=0.2)

In [20]:
model = SVD()

model.fit(trainset)

In [21]:
predictions = model.test(testset)

rmse(predictions)

RMSE: 1.4301


1.4301259309713252

In [22]:
model.predict(1, 'cry')

Prediction(uid=1, iid='cry', r_ui=None, est=2.810053868656542, details={'was_impossible': False})

In [23]:
def recommend_songs(user_id, df, model, top_n=10):

    songs = df['track_name'].unique()

    predictions = []

    for song in songs:

        pred = model.predict(user_id, song)

        predictions.append((song, pred.est))

    predictions.sort(key=lambda x: x[1], reverse=True)

    return predictions[:top_n]

In [24]:
recommendations = recommend_songs(5, df, model)

recommendations

[('dream a little dream', 3.8477991845499635),
 ('hurt so bad', 3.8185892028269146),
 ('a love so beautiful', 3.775865177610827),
 ("it's killing me", 3.768403705935355),
 ('please forgive me', 3.747405938017587),
 ("hurtin' inside", 3.7355260995086814),
 ('revolution', 3.724145992353406),
 ('thanks to you', 3.706370990825435),
 ('if i should lose you', 3.6986170969709358),
 ('i go to sleep', 3.695520485830776)]

In [25]:
def hybrid_recommendation(
    user_id,
    song_title,
    content_scores,
    collaborative_model,
    df,
    top_n=10
):

    hybrid_scores = []

    for idx, row in df.iterrows():

        song = row['track_name']

        content_score = content_scores[idx]

        collab_score = collaborative_model.predict(
            user_id,
            song
        ).est

        final_score = (
            0.5 * content_score
            +
            0.5 * collab_score
        )

        hybrid_scores.append((song, final_score))

    hybrid_scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return hybrid_scores[:top_n]